# LSTM Sentiment Analysis on IMDB Movie Reviews

Every day, millions of reviews, tweets, and comments are posted online.
Businesses need to understand sentiment at scale manualy reading is impossible.

**Goal:** Build an LSTM that reads a movie review and classifies it as
positive or negative, then visualise what the model actually focused on.

**Dataset:** IMDB 50,000 real movie reviews (25k train / 25k test), perfectly balanced.

**Pipeline:**
- Tokenisation: raw text to integer sequences
- Embedding layer: integers to dense 128-dim word vectors
- Bidirectional LSTM: reads sequence forwards and backwards
- Dense + Sigmoid: outputs probability of positive sentiment

In [1]:
#Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, LSTM, Dense,
                                     Dropout, Bidirectional)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow: {tf.__version__}")

TensorFlow: 2.20.0


In [2]:
#Loading the data
VOCAB_SIZE = 10000
MAX_LEN = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f"Train: {len(X_train)} reviews | Test: {len(X_test)} reviews")
print(f"Sample review (raw integers): {X_train[0][:10]}...")
print(f"Label: {y_train[0]} (1=positive, 0=negative)")
print(f"Review lengths — min: {min(len(x) for x in X_train)} | max: {max(len(x) for x in X_train)} | avg: {int(np.mean([len(x) for x in X_train]))}")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train: 25000 reviews | Test: 25000 reviews
Sample review (raw integers): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65]...
Label: 1 (1=positive, 0=negative)
Review lengths — min: 11 | max: 2494 | avg: 238


In [3]:
#Pading all reviews to fixed length of 200 words
X_train = pad_sequences(X_train, maxlen=MAX_LEN, truncating='post', padding='post')
X_test  = pad_sequences(X_test,  maxlen=MAX_LEN, truncating='post', padding='post')

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"\nPositive reviews in train: {y_train.sum()} | Negative: {(y_train==0).sum()}")

X_train shape: (25000, 200)
X_test shape:  (25000, 200)

Positive reviews in train: 12500 | Negative: 12500


In [4]:
#Bidirectional LSTM model: embedding -> BiLSTM -> dropout -> sigmoid output
inputs = Input(shape=(MAX_LEN,))
x = Embedding(VOCAB_SIZE, 128)(inputs)
x = Bidirectional(LSTM(64, return_sequences=False))(x)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,382,977 (5.28 MB)

 Trainable params: 1,382,977 (5.28 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
#lets now train with early stopping
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
    verbose=1
)

Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - accuracy: 0.7131 - loss: 0.5394 - val_accuracy: 0.8434 - val_loss: 0.3877
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.8777 - loss: 0.3095 - val_accuracy: 0.8254 - val_loss: 0.3871
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9158 - loss: 0.2243 - val_accuracy: 0.8662 - val_loss: 0.3721
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9380 - loss: 0.1689 - val_accuracy: 0.8506 - val_loss: 0.4077
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.9591 - loss: 0.1201 - val_accuracy: 0.8454 - val_loss: 0.4444


In [6]:
#Evaluatin on heldout test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Test Loss:     {loss:.4f}")

y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

print("\n", classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Test Accuracy: 84.05%
Test Loss:     0.4320

               precision    recall  f1-score   support

    Negative       0.81      0.89      0.85     12500
    Positive       0.88      0.79      0.83     12500

    accuracy                           0.84     25000
   macro avg       0.84      0.84      0.84     25000
weighted avg       0.84      0.84      0.84     25000



In [9]:
# Now lets predict sentiment on custom reviews

word_index = imdb.get_word_index()

def predict_sentiment(review):
    words = review.lower().split()
    sequence = [word_index.get(w, 0) + 3 for w in words]
    sequence = [i if i < VOCAB_SIZE else 2 for i in sequence]
    padded = pad_sequences([sequence], maxlen=MAX_LEN, padding='post', truncating='post')
    prob = model.predict(padded, verbose=0)[0][0]
    label = 'POSITIVE' if prob >= 0.5 else 'NEGATIVE'
    print(f"Review:     {review[:80]}...")
    print(f"Prediction: {label} ({prob*100:.1f}% confidence)\n")

predict_sentiment("This movie was absolutely fantastic. The acting was briliant and the story kept me hooked till the very end.")
predict_sentiment("Terrible film. Boring plot, bad acting, complete waste of time. I walked out after 20 minutes.")
predict_sentiment("It was okay, nothing special. Some parts were good but overall prety average.")

Review:     This movie was absolutely fantastic. The acting was briliant and the story kept ...
Prediction: POSITIVE (69.0% confidence)

Review:     Terrible film. Boring plot, bad acting, complete waste of time. I walked out aft...
Prediction: NEGATIVE (0.2% confidence)

Review:     It was okay, nothing special. Some parts were good but overall prety average....
Prediction: NEGATIVE (36.5% confidence)



## Key Findings

**Model:** Bidirectional LSTM with learned word embeddings
trained on 25,000 IMDB movie reviews

**Performance:** 84% test accuracy, balanced precision and recall on both classes

**Architecture insight:** 1.28M of 1.38M total parameters are in the Embedding layer.
The actual LSTM reasoning uses only 98K parameters. Word representatins
dominate the parameter count in NLP models.

**Bidirectional advantage:** reading sequences both forwards and backwards
captures context that a unidirectional LSTM misses, particularly for
sentiment words that depend on what comes after them ("not bad", "could have been great").

**Overfitting observed:** training accuracy reached 95% while val accuracy
peaked at 86% by epoch 3. EarlyStopping restored best weights before
the model memorised training reviews.

**Limitation:** model uses a fixed vocabulary of 10,000 words. Slang,
sarcasm, and domain-specific language outside this vocabulary map
to the unknown token and are invisible to the model.